In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql.functions import size
from pyspark.ml.feature import StringIndexer, CountVectorizer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.sql.functions import lower, regexp_replace, col, split
import os

In [2]:
spark = SparkSession.builder \
    .appName("project").getOrCreate()

In [ ]:
df = spark.read.csv(r'.../train.csv', header=True, inferSchema=True)  #add path to train.csv file

In [4]:
def preprocess_data(df):
    df = df.withColumn("text", lower(col("text")))  # Lowercasing
    df = df.withColumn("text", regexp_replace(col("text"), "[^a-zA-Z\\s]", ""))  # Remove punctuation
    emoji_pattern = "[\U00010000-\U0010FFFF]"
    df = df.withColumn("text", regexp_replace(col("text"), emoji_pattern, ""))
    df = df.withColumn("text", split(col("text"), " "))  # Split text into an array of words
    return df

df_cleaned = preprocess_data(df)
df_cleaned = df_cleaned.filter((df_cleaned.text.isNotNull()) & (size(df_cleaned.text) > 0))
df_cleaned = df_cleaned.filter(size(df_cleaned.text) >= 2)

In [5]:
# Split dataset into training and test sets (90-10 split)
train_df, test_df = df_cleaned.randomSplit([0.85, 0.15], seed=42)

In [ ]:
# Step 1: Index categorical labels
indexer = StringIndexer(inputCol="category", outputCol="label", handleInvalid="skip")

# Step 2: Vectorize text data
vectorizer = CountVectorizer(inputCol="text", outputCol="features")

# Step 3: Initialize the Logistic Regression classifier
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Step 4: Create a pipeline
pipeline = Pipeline(stages=[indexer, vectorizer, lr])

# Step 5: Set up the parameter grid for Logistic Regression
paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.01, 0.1, 1.0])  # Adding smaller regParams
             .addGrid(lr.elasticNetParam, [0.25, 0.75])  # Adding finer elasticNet grid
             .build())


# Step 6: Create an evaluator for Logistic Regression
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

# Step 7: Set up CrossValidator
crossval = CrossValidator(estimator=pipeline,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=5)

# Step 8: Train the model using CrossValidator
cv_model = crossval.fit(train_df)

# Step 9: Test the model on the test data
predictions = cv_model.transform(test_df)

# Step 10: Evaluate the accuracy of the model
accuracy = evaluator.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)

print(f"Model Accuracy: {accuracy:.2f}")
print(f"Model Accuracy F1: {f1_score:.2f}")